# NASA C-MAPSS Turbofan Engine RUL Prediction
## Notebook 04: Deep Learning with PyTorch (LSTM & 1D-CNN)

### Objective
In this notebook, we explore temporal sequence architectures for Remaining Useful Life (RUL) estimation:
1. **Sequential Modeling vs Static Aggregations**: Why sequence models preserve temporal degradation dynamics.
2. **PyTorch TurbofanLSTM**: Multi-layer LSTM network with recurrent memory cells.
3. **PyTorch Turbofan1DCNN**: Temporal convolutional neural network extracting local sensor degradation patterns.
4. **Huber / Smooth L1 Loss**: Formulating robust loss functions against noisy outlier cycles.
5. **Comparative Evaluation**: Test RMSE, MAE, R², and official NASA Asymmetric Score.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from config.config import PROCESSED_DATA_DIR, MODELS_DIR, RANDOM_SEED
from src.models import TurbofanLSTM, Turbofan1DCNN
from src.evaluate import evaluate_predictions

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch active device: {device}")

### 1. Load 3D Sequence Windows
Each sample is a 3D matrix of shape `(batch_size, sequence_length=30, features=14)`.

In [ ]:
X_train = np.load(PROCESSED_DATA_DIR / "FD001_X_seq_train.npy")
y_train = np.load(PROCESSED_DATA_DIR / "FD001_y_seq_train.npy")
X_val = np.load(PROCESSED_DATA_DIR / "FD001_X_seq_val.npy")
y_val = np.load(PROCESSED_DATA_DIR / "FD001_y_seq_val.npy")
X_test = np.load(PROCESSED_DATA_DIR / "FD001_X_seq_test.npy")
y_test = np.load(PROCESSED_DATA_DIR / "FD001_y_seq_test.npy")

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

### 2. Inspect Trained Deep Learning Checkpoints
We load the trained weights for `TurbofanLSTM` and `Turbofan1DCNN`.

In [ ]:
lstm_model = TurbofanLSTM(input_dim=14, hidden_dim=64, num_layers=2, dropout=0.2).to(device)
cnn_model = Turbofan1DCNN(input_dim=14, dropout=0.2).to(device)

if (MODELS_DIR / "lstm_model.pt").exists():
    lstm_model.load_state_dict(torch.load(MODELS_DIR / "lstm_model.pt", map_location=device))
    print("Loaded trained LSTM model.")
if (MODELS_DIR / "1dcnn_model.pt").exists():
    cnn_model.load_state_dict(torch.load(MODELS_DIR / "1dcnn_model.pt", map_location=device))
    print("Loaded trained 1D-CNN model.")

### 3. Evaluate Predictions on 100 Test Engines
We predict RUL for all 100 test units and compute evaluation metrics against `RUL_FD001.txt`.

In [ ]:
test_tensor = torch.from_numpy(X_test).to(device)

lstm_model.eval()
cnn_model.eval()
with torch.no_grad():
    lstm_preds = np.clip(lstm_model(test_tensor).cpu().numpy(), 0, None)
    cnn_preds = np.clip(cnn_model(test_tensor).cpu().numpy(), 0, None)

m_lstm = evaluate_predictions(y_test, lstm_preds)
m_lstm["Model"] = "LSTM (PyTorch)"

m_cnn = evaluate_predictions(y_test, cnn_preds)
m_cnn["Model"] = "1D-CNN (PyTorch)"

deep_df = pd.DataFrame([m_lstm, m_cnn])[["Model", "RMSE", "MAE", "R2", "NASA_Score", "Early_Predictions", "Late_Predictions"]]
deep_df

### 4. Plot Test Engine Predictions vs Ground Truth
We compare how the LSTM tracks the true remaining operational cycles of test engines.

In [ ]:
sort_idx = np.argsort(y_test)
plt.figure(figsize=(12, 5))
plt.plot(np.arange(100), y_test[sort_idx], label="Actual Ground Truth RUL", color="black", linewidth=2.2, linestyle="--")
plt.plot(np.arange(100), lstm_preds[sort_idx], label="LSTM Prediction", color="#e74c3c", linewidth=1.8)
plt.plot(np.arange(100), cnn_preds[sort_idx], label="1D-CNN Prediction", color="#8e44ad", linewidth=1.5, alpha=0.8)
plt.title("Deep Learning Test Predictions vs Ground Truth (Sorted by Actual RUL)")
plt.xlabel("Test Engines (Ranked by Ascending Actual Lifespan)")
plt.ylabel("Remaining Useful Life (Cycles)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()